# Data Collection and Processing

In [5]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
# Load the downloaded csv from Kaggle
df = pd.read_csv('../raw_data/bot_detection_data.csv')
print(f"Columns:{df.columns.tolist()}")
len(df)

Columns:['User ID', 'Username', 'Tweet', 'Retweet Count', 'Mention Count', 'Follower Count', 'Verified', 'Bot Label', 'Location', 'Created At', 'Hashtags']


50000

### Create the relational database tables

In [15]:
# Table 1: users
users = df[['User ID', 'Username', 'Follower Count', 'Verified', 'Location', 'Bot Label']].drop_duplicates(subset='User ID')
users.columns = ['user_id', 'username', 'follower_count', 'verified', 'location', 'bot_label']
users.to_csv('users.csv', index=False)

# Table 2: tweets
tweets = df[['User ID', 'Tweet', 'Retweet Count', 'Mention Count', 'Created At']].copy()
tweets.insert(0, 'Tweet ID', range(1, len(tweets) + 1))
tweets.columns = ['tweet_id', 'user_id', 'tweet', 'retweet_count', 'mention_count', 'created_at']
tweets.to_csv('tweets.csv', index=False)

# Table 3: hashtags (one row per hashtag per tweet)
tweets_temp = tweets[['tweet_id']].copy()
hashtags_raw = df['Hashtags'].copy()
hashtags_df = pd.DataFrame({
    'tweet_id': tweets_temp['tweet_id'],
    'hashtags': hashtags_raw
})
hashtags_df = hashtags_df.dropna(subset=['hashtags'])
hashtags_df = hashtags_df.assign(hashtag=hashtags_df['hashtags'].str.split()).explode('hashtag')
hashtags_df = hashtags_df[['tweet_id', 'hashtag']].reset_index(drop=True)
hashtags_df.insert(0, 'hashtag_id', range(1, len(hashtags_df) + 1))
hashtags_df.to_csv('hashtags.csv', index=False)

# Table 4: locations (lookup table)
locations = df[['Location']].drop_duplicates().dropna().reset_index(drop=True)
locations.columns = ['location']
locations.insert(0, 'location_id', range(1, len(locations) + 1))

# add location_id back to users table
users = users.merge(locations, on='location', how='left')
users.to_csv('users.csv', index=False)
locations.to_csv('locations.csv', index=False)

In [20]:
# Take a look at the tables
print(users.head())
print(tweets.head())
print(hashtags_df.head())
print(locations.head())
print("\n")
print(len(users))
print(len(tweets))
print(len(hashtags_df))
print(len(locations))

   user_id        username  follower_count  verified      location  bot_label  \
0   132131           flong            2353     False     Adkinston          1   
1   289683  hinesstephanie            9617      True    Sanderston          0   
2   779715      roberttran            4363      True  Harrisonfurt          0   
3   696168          pmason            2242      True  Martinezberg          1   
4   704441          noah87            8438     False  Camachoville          1   

   location_id  
0            1  
1            2  
2            3  
3            4  
4            5  
   tweet_id  user_id                                              tweet  \
0         1   132131  Station activity person against natural majori...   
1         2   289683  Authority research natural life material staff...   
2         3   779715  Manage whose quickly especially foot none to g...   
3         4   696168  Just cover eight opportunity strong policy which.   
4         5   704441                

In [3]:
import zipfile
with zipfile.ZipFile('../raw_data/tweet-bot-data.zip', 'r') as z:
    print(z.namelist())

['tweet-bot-data/', 'tweet-bot-data/bot/', 'tweet-bot-data/bot/tweets.csv', 'tweet-bot-data/bot/users.csv', 'tweet-bot-data/real/', 'tweet-bot-data/real/tweets.csv', 'tweet-bot-data/real/users.csv']


In [ ]:
with zipfile.ZipFile('../raw_data/tweet-bot-data.zip', 'r') as z:
    with z.open('tweet-bot-data/real/users.csv') as f:
        real_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/users.csv') as f:
        bot_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/real/tweets.csv') as f:
        real_tweets = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/tweets.csv') as f:
        bot_tweets = pd.read_csv(f, encoding='latin-1')

/tmp/ipykernel_147451/4151748497.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  real_tweets = pd.read_csv(f, encoding='latin-1')
/tmp/ipykernel_147451/4151748497.py:9: DtypeWarning: Columns (8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  bot_tweets = pd.read_csv(f, encoding='latin-1')


In [8]:
print("=== REAL USERS ===")
print(real_users.columns.tolist())
print(real_users.shape)

print("\n=== BOT USERS ===")
print(bot_users.columns.tolist())
print(bot_users.shape)

print("\n=== REAL TWEETS ===")
print(real_tweets.columns.tolist())
print(real_tweets.shape)

print("\n=== BOT TWEETS ===")
print(bot_tweets.columns.tolist())
print(bot_tweets.shape)

=== REAL USERS ===
['id', 'name', 'screen_name', 'statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'url', 'lang', 'time_zone', 'location', 'default_profile', 'default_profile_image', 'geo_enabled', 'profile_image_url', 'profile_banner_url', 'profile_use_background_image', 'profile_background_image_url_https', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_border_color', 'profile_background_tile', 'profile_sidebar_fill_color', 'profile_background_image_url', 'profile_background_color', 'profile_link_color', 'utc_offset', 'is_translator', 'follow_request_sent', 'protected', 'verified', 'notifications', 'description', 'contributors_enabled', 'following', 'created_at', 'timestamp', 'crawled_at', 'updated', 'test_set_1', 'test_set_2']
(3474, 42)

=== BOT USERS ===
['id', 'name', 'screen_name', 'statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'created_at', 'url', 'lang', 'time_zone', 'location